In [1]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from openai import OpenAI
import joblib

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [2]:
# 1. Load Data from JSONL files (using 'originator' and source metadata)
def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

def jsonl_to_df(path):
    data = load_jsonl(path)
    # Use 'originator' as the source identifier
    df = pd.DataFrame(data)
    # If 'originator' is missing, fill with 'unknown'
    if 'originator' not in df.columns:
        df['originator'] = 'unknown'
    return df[['originator']]

train_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/train2.jsonl')
val_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/val2.jsonl')
test_df = jsonl_to_df('/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/Liar-Plus-Recreation/Data/LIAR+ datasets/test2.jsonl')

In [3]:
# 2. Source Reputation Feature Engineering
source_metadata = {
    'dwayne-bohac': {'accuracy_score': 0.4, 'awards': 0, 'editorial_transparency': 'low'},
    'scott-surovell': {'accuracy_score': 0.7, 'awards': 1, 'editorial_transparency': 'medium'},
    'barack-obama': {'accuracy_score': 0.9, 'awards': 2, 'editorial_transparency': 'high'},
    'blog-posting': {'accuracy_score': 0.2, 'awards': 0, 'editorial_transparency': 'unknown'},
}

transparency_map = {'low': 0, 'medium': 1, 'high': 2, 'unknown': -1, None: -1}

for df in [train_df, val_df, test_df]:
    df['source_accuracy'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('accuracy_score', -1))
    df['source_awards'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('awards', -1))
    df['source_editorial_transparency_num'] = df['originator'].map(
        lambda x: transparency_map.get(source_metadata.get(x, {}).get('editorial_transparency', 'unknown'), -1)
    )

In [4]:
# 4. Generate Embeddings
def get_embedding(text: str, api_key: str = OPENROUTER_API_KEY, model: str = "google/gemini-embedding-001") -> list[float]:
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    embedding = client.embeddings.create(model=model, input=text, encoding_format="float")
    return embedding.data[0].embedding

def embed_column(df: pd.DataFrame, column: str, api_key: str = OPENROUTER_API_KEY, model: str = "google/gemini-embedding-001", max_workers: int = 10) -> pd.DataFrame:
    df_result = df.copy()
    texts = df[column].astype(str).tolist()

    def embed_single(text):
        try:
            return get_embedding(text, api_key=api_key, model=model)
        except Exception as e:
            print(f"Exception on: {text[:50]} -- {str(e)}")
            return None

    embeddings = [None] * len(texts)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(embed_single, text): idx for idx, text in enumerate(texts)}
        for future in tqdm(as_completed(future_to_idx), total=len(texts), desc=f"Embedding {column}"):
            idx = future_to_idx[future]
            embeddings[idx] = future.result()

    # Determine embedding dimension
    embedding_dim = next((len(e) for e in embeddings if e is not None), 0)
    for dim in range(embedding_dim):
        df_result[f"{column}_embedding_{dim}"] = [e[dim] if e is not None else np.nan for e in embeddings]

    return df_result

In [5]:
# 4b. Generate embeddings for originator with sampling and optional caching.
# Sample the train split to avoid generating embeddings for the entire training set.
TRAIN_SAMPLE_SIZE = 2000
ENABLE_CACHE = True
CACHE_DIR = os.path.join(os.getcwd(), 'embeddings_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_path(split_name):
    return os.path.join(CACHE_DIR, f"{split_name}_originator_embeddings.pkl")

def _merge_embeddings(target_df, embedded_df, prefix='originator_embedding_'):
    cols = [c for c in embedded_df.columns if c.startswith(prefix)]
    if not cols:
        return target_df
    return pd.concat([target_df, embedded_df[cols]], axis=1)

def _embeddings_to_df(df, embeddings, prefix='originator_embedding_'):
    if len(embeddings) == 0:
        return pd.DataFrame(index=df.index)
    embedding_dim = len(next(e for e in embeddings if e is not None))
    embeddings_array = np.array([e if e is not None else [np.nan]*embedding_dim for e in embeddings])
    columns = [f"{prefix}{i}" for i in range(embedding_dim)]
    return pd.DataFrame(embeddings_array, columns=columns, index=df.index)

# TRAIN
train_cache = _cache_path('train')
if ENABLE_CACHE and os.path.exists(train_cache):
    train_embedded = pd.read_pickle(train_cache)
    train_df = _merge_embeddings(train_df, train_embedded)
else:
    train_sample = train_df.sample(n=min(TRAIN_SAMPLE_SIZE, len(train_df)), random_state=42)
    train_embeddings = embed_column(train_sample, 'originator')
    train_embedded_df = _embeddings_to_df(train_sample, train_embeddings)
    train_df = _merge_embeddings(train_df, train_embedded_df)
    if ENABLE_CACHE:
        pd.to_pickle(train_embedded_df, train_cache)

# VAL
val_cache = _cache_path('val')
if ENABLE_CACHE and os.path.exists(val_cache):
    val_embedded = pd.read_pickle(val_cache)
    val_df = _merge_embeddings(val_df, val_embedded)
else:
    val_embeddings = embed_column(val_df, 'originator')
    val_embedded_df = _embeddings_to_df(val_df, val_embeddings)
    val_df = _merge_embeddings(val_df, val_embedded_df)
    if ENABLE_CACHE:
        pd.to_pickle(val_embedded_df, val_cache)

# TEST
test_cache = _cache_path('test')
if ENABLE_CACHE and os.path.exists(test_cache):
    test_embedded = pd.read_pickle(test_cache)
    test_df = _merge_embeddings(test_df, test_embedded)
else:
    test_embeddings = embed_column(test_df, 'originator')
    test_embedded_df = _embeddings_to_df(test_df, test_embeddings)
    test_df = _merge_embeddings(test_df, test_embedded_df)
    if ENABLE_CACHE:
        pd.to_pickle(test_embedded_df, test_cache)


In [6]:
# 5. Encode Source Reputation Labels (for clustering or classification)
le = LabelEncoder()
train_df['source_label'] = le.fit_transform(train_df['originator'])

def safe_transform(le, values):
    classes = set(le.classes_)
    return [le.transform([v])[0] if v in classes else -1 for v in values]

val_df['source_label'] = safe_transform(le, val_df['originator'])
test_df['source_label'] = safe_transform(le, test_df['originator'])

# Keep only known classes in val/test
val_df_clean = val_df[val_df['source_label'] != -1].copy()
test_df_clean = test_df[test_df['source_label'] != -1].copy()


/var/folders/lh/4mq6w_g54p5d9v09ngbytmzc0000gn/T/ipykernel_47868/2445768212.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['source_label'] = le.fit_transform(train_df['originator'])


In [7]:
# 6. Prepare Features and Labels for Source Reputation
feature_cols = ['source_accuracy', 'source_awards', 'source_editorial_transparency_num']

X_train = train_df[feature_cols].to_numpy()
y_train = train_df['source_label'].to_numpy()
X_val = val_df_clean[feature_cols].to_numpy()
y_val = val_df_clean['source_label'].to_numpy()
X_test = test_df_clean[feature_cols].to_numpy()
y_test = test_df_clean['source_label'].to_numpy()

print('Feature matrix shapes — X_train:', X_train.shape, 'X_val:', X_val.shape, 'X_test:', X_test.shape)


Feature matrix shapes — X_train: (10269, 3) X_val: (1063, 3) X_test: (1083, 3)


In [8]:
# 7. Train Only Random Forest Classifier for Source Reputation
clf = RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1, random_state=42)
clf.fit(X_train, y_train)

# Validation
y_pred_val = clf.predict(X_val)
valid_labels = np.unique(y_val)
report = classification_report(
    y_val, y_pred_val,
    labels=valid_labels,
    target_names=[le.classes_[i] for i in valid_labels]
)
acc = accuracy_score(y_val, y_pred_val)
print(f"Random Forest Validation Accuracy: {acc:.4f}")
print(report)

# Test predictions
y_pred_test = clf.predict(X_test)
pred_labels = le.inverse_transform(y_pred_test)
test_df_clean_result = test_df_clean[['originator', 'source_accuracy', 'source_awards', 'source_editorial_transparency_num']].copy()
test_df_clean_result['predicted_source'] = pred_labels
print(test_df_clean_result.head())


Random Forest Validation Accuracy: 0.1035
                                             precision    recall  f1-score   support

                        60-plus-association       0.00      0.00      0.00         1
                                    afl-cio       0.00      0.00      0.00         1
                                     afscme       0.00      0.00      0.00         1
                               alan-grayson       0.00      0.00      0.00         2
                                  alex-sink       0.00      0.00      0.00         1
                                 allan-fung       0.00      0.00      0.00         1
                                allen-peake       0.00      0.00      0.00         1
                                 allen-west       0.00      0.00      0.00         1
                               allison-tant       0.00      0.00      0.00         1
                               amanda-fritz       0.00      0.00      0.00         1
                      

/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precisi

In [9]:
# 6. Example Output: Show predictions for first few sources (Random Forest only)
# Predict on test set
y_pred_test = clf.predict(X_test)
pred_labels = le.inverse_transform(y_pred_test)

# Show first few predictions
test_df_clean_result = test_df_clean[['originator', 'source_accuracy', 'source_awards', 'source_editorial_transparency_num']].copy()
test_df_clean_result['predicted_source'] = pred_labels
print(test_df_clean_result.head())


                         originator  source_accuracy  source_awards  \
0                        rick-perry             -1.0             -1   
1                 katrina-shankland             -1.0             -1   
2                      donald-trump             -1.0             -1   
3                     rob-cornilles             -1.0             -1   
4  state-democratic-party-wisconsin             -1.0             -1   

   source_editorial_transparency_num predicted_source  
0                                 -1     donald-trump  
1                                 -1     donald-trump  
2                                 -1     donald-trump  
3                                 -1     donald-trump  
4                                 -1     donald-trump  


In [10]:
# # Prepare features and labels (Random Forest only, source reputation features only)
# X_train = train_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
# y_train = train_df['source_label'].to_numpy()
# X_val = val_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
# y_val = val_df['source_label'].to_numpy()
# X_test = test_df[['source_accuracy', 'source_awards', 'source_transparency']].to_numpy()
# y_test = test_df['source_label'].to_numpy()

# results = {}
# for name, clf in classifiers.items():
#     clf.fit(X_train, y_train)
#     y_pred_val = clf.predict(X_val)
#     acc = accuracy_score(y_val, y_pred_val)
#     # Get unique labels in y_val that are not -1 (unseen)
#     valid_labels = np.unique(y_val[y_val != -1])
#     valid_class_names = le.inverse_transform(valid_labels)
#     report = classification_report(
#         y_val, y_pred_val,
#         labels=valid_labels,
#         target_names=valid_class_names
#     )
#     results[name] = (acc, report)
#     print(f"Classifier: {name}")
#     print(f"Validation Accuracy: {acc:.4f}")
#     print("Classification Report:")
#     print(report)
#     print("-" * 60)

# # Output predictions for test set (Random Forest only, source reputation features only)
# for name, clf in classifiers.items():
#     y_pred_test = clf.predict(X_test)
#     pred_labels = le.inverse_transform(y_pred_test)
#     print(f"Classifier: {name}")
#     print(test_df[['originator', 'source_accuracy', 'source_awards', 'source_transparency']].assign(predicted_source=pred_labels).head())
#     print("-" * 60)

In [11]:
# --- 9. Train and Save XGBoost Source Reputation Model ---
TARGET_MODEL_DIR = '/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/model_training_scripts/source_reputation'
os.makedirs(TARGET_MODEL_DIR, exist_ok=True)

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss",
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    seed=42
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)

y_pred_val = xgb_model.predict(X_val)
acc = accuracy_score(y_val, y_pred_val)
print(f"XGBoost Validation Accuracy: {acc:.4f}")
print(classification_report(y_val, y_pred_val))

model_path = os.path.join(TARGET_MODEL_DIR, 'source_reputation_model.gz')
joblib.dump(xgb_model, model_path, compress=3)
print(f"✅ Saved XGBoost source reputation model to {model_path}")


[0]	validation_0-mlogloss:7.21867
[1]	validation_0-mlogloss:7.03217
[1]	validation_0-mlogloss:7.03217
[2]	validation_0-mlogloss:6.90103
[2]	validation_0-mlogloss:6.90103
[3]	validation_0-mlogloss:6.80166
[3]	validation_0-mlogloss:6.80166
[4]	validation_0-mlogloss:6.70000
[4]	validation_0-mlogloss:6.70000
[5]	validation_0-mlogloss:6.61056
[5]	validation_0-mlogloss:6.61056
[6]	validation_0-mlogloss:6.54881
[6]	validation_0-mlogloss:6.54881
[7]	validation_0-mlogloss:6.49557
[7]	validation_0-mlogloss:6.49557
[8]	validation_0-mlogloss:6.45002
[8]	validation_0-mlogloss:6.45002
[9]	validation_0-mlogloss:6.41091
[9]	validation_0-mlogloss:6.41091
[10]	validation_0-mlogloss:6.37541
[10]	validation_0-mlogloss:6.37541
[11]	validation_0-mlogloss:6.34359
[11]	validation_0-mlogloss:6.34359
[12]	validation_0-mlogloss:6.31486
[12]	validation_0-mlogloss:6.31486
[13]	validation_0-mlogloss:6.28984
[13]	validation_0-mlogloss:6.28984
[14]	validation_0-mlogloss:6.26655
[14]	validation_0-mlogloss:6.26655
[15]

/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precisi

✅ Saved XGBoost source reputation model to /Users/danielbirman/Desktop/UCSD/Year 4/Fall 2025/DSC 180A/capstone_factuality_factors/model_training_scripts/source_reputation/source_reputation_model.gz


In [12]:
# --- Source Reputation Metadata Analysis ---
# This cell demonstrates how to integrate source metadata for reputation analysis.

# Example: Define source metadata (in practice, load from a file/database)
source_metadata = {
    # originator: {accuracy_score, awards, editorial_transparency}
    'dwayne-bohac': {'accuracy_score': 0.4, 'awards': 0, 'editorial_transparency': 'low'},
    'scott-surovell': {'accuracy_score': 0.7, 'awards': 1, 'editorial_transparency': 'medium'},
    'barack-obama': {'accuracy_score': 0.9, 'awards': 2, 'editorial_transparency': 'high'},
    'blog-posting': {'accuracy_score': 0.2, 'awards': 0, 'editorial_transparency': 'unknown'},
    # ... add more sources as needed ...
}

# Map metadata to each claim in the training/validation/test sets
for df in [train_df, val_df, test_df]:
    if 'originator' in df.columns:
        df['source_accuracy'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('accuracy_score', None))
        df['source_awards'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('awards', None))
        df['source_editorial_transparency'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('editorial_transparency', None))
    else:
        # If 'originator' column is missing, fill with default values
        df['source_accuracy'] = -1
        df['source_awards'] = -1
        df['source_editorial_transparency'] = 'unknown'

# Example: Print summary statistics for source reputation features
print('Source accuracy scores:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_accuracy']].drop_duplicates().head())
else:
    print(train_df[['source_accuracy']].drop_duplicates().head())
print('\nSource awards:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_awards']].drop_duplicates().head())
else:
    print(train_df[['source_awards']].drop_duplicates().head())
print('\nSource editorial transparency:')
if 'originator' in train_df.columns:
    print(train_df[['originator', 'source_editorial_transparency']].drop_duplicates().head())
else:
    print(train_df[['source_editorial_transparency']].drop_duplicates().head())

# Example: Use these features in model training (add to X_train, X_val, X_test)
# For demonstration, fill missing values with -1 or 'unknown' and encode transparency
transparency_map = {'low': 0, 'medium': 1, 'high': 2, 'unknown': -1, None: -1}
for df in [train_df, val_df, test_df]:
    df['source_accuracy'] = df['source_accuracy'].fillna(-1)
    df['source_awards'] = df['source_awards'].fillna(-1)
    df['source_editorial_transparency_num'] = df['source_editorial_transparency'].map(transparency_map)

# Update feature matrices to include new metadata (Random Forest only)
X_train = train_df[['source_accuracy', 'source_awards', 'source_editorial_transparency_num']].to_numpy()
y_train = train_df['source_label'].to_numpy()
X_val = val_df[['source_accuracy', 'source_awards', 'source_editorial_transparency_num']].to_numpy()
y_val = val_df['source_label'].to_numpy()
X_test = test_df[['source_accuracy', 'source_awards', 'source_editorial_transparency_num']].to_numpy()
y_test = test_df['source_label'].to_numpy()

print('Updated feature matrix shape:', X_train.shape)

Source accuracy scores:
       originator  source_accuracy
0    dwayne-bohac              0.4
1  scott-surovell              0.7
2    barack-obama              0.9
3    blog-posting              0.2
4   charlie-crist              NaN

Source awards:
       originator  source_awards
0    dwayne-bohac            0.0
1  scott-surovell            1.0
2    barack-obama            2.0
3    blog-posting            0.0
4   charlie-crist            NaN

Source editorial transparency:
       originator source_editorial_transparency
0    dwayne-bohac                           low
1  scott-surovell                        medium
2    barack-obama                          high
3    blog-posting                       unknown
4   charlie-crist                          None
Updated feature matrix shape: (10269, 3)


/var/folders/lh/4mq6w_g54p5d9v09ngbytmzc0000gn/T/ipykernel_47868/3269978592.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['source_editorial_transparency'] = df['originator'].map(lambda x: source_metadata.get(x, {}).get('editorial_transparency', None))
